## Session 4 · Clean and Transform — the 1500m data (Python)

**Dataset:** `w1500_alltime_fields.csv` — the all-time women's 1500m performances
you web-scraped in Session 2.

This is the **Clean** and **Transform** stages of the lifecycle. The scrape is raw:
the time is text, the dates are text, and a two-digit birth year hides a trap. We
fix all of that, derive the columns we need, then save a cleaned file for the next
notebooks. Run each cell in order.

### 1. Load the raw data

Read the CSV into `df`, then print its shape, dtypes, and first rows.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_FOLDER = Path("../data")
DATA_FILE = DATA_FOLDER / "w1500_alltime_fields.csv"
CLEAN_FILE = DATA_FOLDER / "w1500_clean.csv"

df = pd.read_csv(DATA_FILE)
print(df.shape)
df.head()

### 2. Inspect the problems

Print the column types and the count of missing values. Which columns are text that should be numbers or dates?

In [ ]:
print(df.dtypes)
print(df.isna().sum())

### 3. Clean the `mark` column

The time is written as text like `3:48.68`. Some marks end in `+` (a time set
*en route* to a longer race) or `A` (a result recorded at altitude).

`endswith()` checks the final character. `clean[:-1]` keeps everything except
that character. `map(to_seconds)` applies the function to every value in the column.

In [ ]:
def to_seconds(mark):
    clean = str(mark).strip()

    if clean.endswith("+"):
        clean = clean[:-1]

    if clean.endswith("A"):
        clean = clean[:-1]

    minutes_text, seconds_text = clean.split(":")
    minutes = int(minutes_text)
    seconds = float(seconds_text)

    return minutes * 60 + seconds


df["mark_seconds"] = df["mark"].map(to_seconds)
df[["mark", "mark_seconds"]].head()

### 4. Parse the dates

Both dates are text. `pd.to_datetime()` converts them to dates. The `format`
describes their order: `%d` is day, `%m` is month, `%Y` is a four-digit year,
and `%y` is a two-digit year. Python places two-digit years in the range
1969–2068, so the printed maximum exposes the century problem.
`errors="coerce"` changes an invalid date to `NaT` instead of stopping the notebook.

In [ ]:
df["result_date"] = pd.to_datetime(df["result_date"], format="%d.%m.%Y", errors="coerce")
df["date_of_birth"] = pd.to_datetime(df["date_of_birth"], format="%d.%m.%y", errors="coerce")
print(df["date_of_birth"].min(), df["date_of_birth"].max())

### 5. Fix the two-digit years

A birth date after its race date must have been placed in the wrong century.
The comparison creates a `True`/`False` mask called `future_birth`.

`df.loc[rows, column]` selects one column from the rows where the mask is `True`.
`pd.offsets.DateOffset(years=100)` represents 100 calendar years.

In [ ]:
future_birth = df["date_of_birth"] > df["result_date"]
df.loc[future_birth, "date_of_birth"] = (
    df.loc[future_birth, "date_of_birth"]
    - pd.offsets.DateOffset(years=100)
)
print("birth dates corrected:", future_birth.sum())

### 6. Transform — derive the analysis columns

`.dt.year` extracts the year from each date. `// 10 * 10` rounds a year down to
the start of its decade. Subtracting two dates gives a duration; `.dt.days`
extracts its number of days, which we divide by `365.25` to estimate age in years.

In [ ]:
df["result_year"] = df["result_date"].dt.year
df["decade"] = df["result_year"] // 10 * 10
df["age_at_race"] = (df["result_date"] - df["date_of_birth"]).dt.days / 365.25
df[["athlete", "mark_seconds", "result_year", "decade", "age_at_race"]].head()

### 7. Sanity-check your work

Run `describe()` on `mark_seconds` and `age_at_race`. Focus on the minimum and
maximum values: do all the ages look believable?

In [ ]:
print(df["mark_seconds"].describe().round(2))
print(df["age_at_race"].describe().round(1))

The maximum suggests a data problem. Is it believable that an athlete ran an
elite 1500 m race at that age? The century rule cannot repair an incorrect source
record. Filter for ages outside an expected range so you can inspect them.

`|` means **or**. Parentheses are needed around each comparison.

In [ ]:
suspicious_age = (df["age_at_race"] < 15) | (df["age_at_race"] > 45)
df.loc[suspicious_age, ["athlete", "date_of_birth", "result_date", "age_at_race"]]

### 8. Remove the suspicious records

The two rows above contain incorrect source data. `~suspicious_age` reverses the
mask so that `True` means keep the row. `.copy()` creates the cleaned DataFrame.

In [ ]:
df = df.loc[~suspicious_age].copy()
print("rows remaining:", len(df))

### 9. Save the cleaned data

Write the cleaned DataFrame to `data/w1500_clean.csv` so the next notebooks can load it.

In [ ]:
df.to_csv(CLEAN_FILE, index=False)
print("saved")

### 10. Wrap-up

In 2–3 sentences: which fixes were **cleaning** and which were **transforming**? Which single problem would have silently ruined an age analysis if you had not checked?